In [ ]:
import torch

In [ ]:
%%writefile model.py
import torch
import torch.nn as nn
from torchvision import models

class ImageOnlyModel(nn.Module):
    def __init__(self, model_name):
        super(ImageOnlyModel, self).__init__()
        self.model_name = model_name.lower()

        if 'efficientnet' in self.model_name:
            if 'b5' in self.model_name:
                self.backbone = models.efficientnet_b5(weights='DEFAULT')
            elif 'b6' in self.model_name:
                self.backbone = models.efficientnet_b6(weights='DEFAULT')
            elif 'b7' in self.model_name:
                self.backbone = models.efficientnet_b7(weights='DEFAULT')
            self.feature_dim = self.backbone.classifier[1].in_features
            self.backbone.classifier = nn.Identity()

        elif 'resnet' in self.model_name:
            if '50' in self.model_name:
                self.backbone = models.resnet50(weights='DEFAULT')
            elif '101' in self.model_name:
                self.backbone = models.resnet101(weights='DEFAULT')
            elif '152' in self.model_name:
                self.backbone = models.resnet152(weights='DEFAULT')
            self.feature_dim = self.backbone.fc.in_features
            self.backbone.fc = nn.Identity()

        elif 'densenet' in self.model_name:
            if '121' in self.model_name:
                self.backbone = models.densenet121(weights='DEFAULT')
            elif '161' in self.model_name:
                self.backbone = models.densenet161(weights='DEFAULT')
            elif '169' in self.model_name:
                self.backbone = models.densenet169(weights='DEFAULT')
            elif '201' in self.model_name:
                self.backbone = models.densenet201(weights='DEFAULT')
            self.feature_dim = self.backbone.classifier.in_features
            self.backbone.classifier = nn.Identity()

        elif 'mobilenet' in self.model_name:
            if 'large' in self.model_name:
                self.backbone = models.mobilenet_v3_large(weights='DEFAULT')
            else:
                self.backbone = models.mobilenet_v3_small(weights='DEFAULT')
            self.feature_dim = self.backbone.classifier[0].in_features
            self.backbone.classifier = nn.Identity()

        elif 'inception' in self.model_name:
            self.backbone = models.inception_v3(weights='DEFAULT')
            self.backbone.aux_logits = False
            self.feature_dim = self.backbone.fc.in_features
            self.backbone.fc = nn.Identity()

        else:
            raise ValueError(f"Model {model_name} not implemented.")

        for param in self.backbone.parameters():
            param.requires_grad = False

        self.classifier = nn.Sequential(
            nn.Linear(self.feature_dim, 1024),
            nn.ReLU(),
            nn.Dropout(0.25),
            nn.Linear(1024, 512),
            nn.ReLU(),
            nn.Dropout(0.25),
            nn.Linear(512, 1)
        )

    def forward(self, x):
        if 'inception' in self.model_name and self.training:
            features = self.backbone(x)
        else:
            features = self.backbone(x)

        if len(features.shape) > 2:
             features = torch.flatten(features, 1)
        return self.classifier(features)

Writing model.py


In [ ]:
%%writefile dataset_utils.py
import os
import pandas as pd
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import torch
import numpy as np

IMAGE_COLUMN = "DDI_file"
LABEL_COLUMN = "malignant"

MODEL_RESOLUTIONS = {
    "resnet50": 224, "resnet101": 224, "resnet152": 224,
    "densenet121": 224, "densenet161": 224, "densenet169": 224, "densenet201": 224,
    "mobilenet_v3_small": 224, "mobilenet_v3_large": 224,
    "inception_v3": 299,
    "efficientnet_b5": 456, "efficientnet_b6": 528, "efficientnet_b7": 600
}

MEAN = [0.485, 0.456, 0.406]
STD  = [0.229, 0.224, 0.225]

class DDIDataset(Dataset):
    def __init__(self, csv_file, img_dir, transform=None, skin_tone_max=None):
        """
+        csv_file: path to metadata CSV
+        skin_tone_max: if not None, keep only rows where 'skin_tone' < skin_tone_max
+        """
        self.data = pd.read_csv(csv_file)
        self.img_dir = img_dir
        self.transform = transform

        if IMAGE_COLUMN not in self.data.columns:
            raise KeyError(f"Column '{IMAGE_COLUMN}' not found. Columns are: {list(self.data.columns)}")

        # optional skin-tone filtering (keep rows with skin_tone < skin_tone_max)
        if skin_tone_max is not None:
            if 'skin_tone' not in self.data.columns:
                raise KeyError("Column 'skin_tone' not found in CSV, cannot apply skin_tone_max filter.")
            # keep rows with skin_tone < skin_tone_max
            self.data = self.data[self.data['skin_tone'] < skin_tone_max].reset_index(drop=True)
            if len(self.data) == 0:
                raise ValueError(f"No samples found after applying skin_tone_max={skin_tone_max} filter.")

        self.image_names = self.data[IMAGE_COLUMN].values

        if self.data[LABEL_COLUMN].dtype == 'bool':
            self.labels = self.data[LABEL_COLUMN].astype(float).values
        else:
          self.labels = self.data[LABEL_COLUMN].values.astype(float)
 # ...existing code...
    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        img_name = self.image_names[idx]
        img_path = os.path.join(self.img_dir, str(img_name))

        if not os.path.exists(img_path):
             if os.path.exists(img_path + '.png'): img_path += '.png'
             elif os.path.exists(img_path + '.jpg'): img_path += '.jpg'

        try:
            image = Image.open(img_path).convert('RGB')
        except (FileNotFoundError, OSError):
            print(f"Warning: Could not find {img_path}, using black image.")
            image = Image.new('RGB', (224, 224))

        label = self.labels[idx]

        if self.transform:
            image = self.transform(image)

        return image, label

def get_transforms(model_name, split='train'):
    size = MODEL_RESOLUTIONS.get(model_name.lower(), 224)
    if split == 'train':
        return transforms.Compose([
            transforms.Resize((size + 32, size + 32)),
            transforms.RandomCrop((size, size)),
            transforms.RandomHorizontalFlip(),
            transforms.RandomRotation(15),
            transforms.ColorJitter(contrast=0.2),
            transforms.ToTensor(),
            transforms.Normalize(mean=MEAN, std=STD)
        ])
    else:
        return transforms.Compose([
            transforms.Resize((size, size)),
            transforms.ToTensor(),
            transforms.Normalize(mean=MEAN, std=STD)
        ])

def get_dataloaders(model_name, data_dir, csv_path, batch_size=8, skin_tone_max=40):
    """
    skin_tone_max: keep only samples with skin_tone < skin_tone_max (default 40)
    """
    train_transforms = get_transforms(model_name, split='train')
    val_transforms   = get_transforms(model_name, split='val')

    full_dataset = DDIDataset(csv_path, data_dir, transform=None, skin_tone_max=skin_tone_max)

    if len(full_dataset) == 0:
        raise ValueError("Dataset is empty after filtering. Check your CSV, skin_tone_max and image directory.")

    train_size = int(0.8 * len(full_dataset))
    val_size = len(full_dataset) - train_size

    train_dataset, val_dataset = torch.utils.data.random_split(full_dataset, [train_size, val_size])

    train_dataset.dataset.transform = train_transforms
    val_dataset.dataset.transform = val_transforms

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=0)
    val_loader   = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=0)

    return train_loader, val_loader

Writing dataset_utils.py


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
%%writefile train_image_only.py
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
from sklearn.metrics import roc_auc_score
from model import ImageOnlyModel
from dataset_utils import get_dataloaders

class BCEWithLogitsLossSmoothing(nn.Module):
    def __init__(self, smoothing=0.0):
        super(BCEWithLogitsLossSmoothing, self).__init__()
        self.smoothing = smoothing
        self.bce = nn.BCEWithLogitsLoss()

    def forward(self, input, target):
        if self.smoothing > 0:
            with torch.no_grad():
                target = target * (1.0 - self.smoothing) + 0.5 * self.smoothing
        return self.bce(input, target)

MODEL_NAME = "efficientnet_b5"
DATA_DIR = "/content/drive/MyDrive/DDI/images"
CSV_PATH = "/content/drive/MyDrive/DDI/ddi_metadata.csv"
BATCH_SIZE = 8
LR = 1e-4
EPOCHS = 100

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

print(f"Preparing data for {MODEL_NAME}...")
train_loader, val_loader = get_dataloaders(MODEL_NAME, DATA_DIR, CSV_PATH, BATCH_SIZE)

model = ImageOnlyModel(MODEL_NAME).to(device)

criterion = BCEWithLogitsLossSmoothing(smoothing=0.1)
optimizer = optim.Adam(model.parameters(), lr=LR)

def evaluate(model, loader):
    model.eval()
    all_targets = []
    all_probs = []
    running_loss = 0.0

    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device).float()
            outputs = model(images).squeeze()

            loss = criterion(outputs, labels)
            running_loss += loss.item()

            probs = torch.sigmoid(outputs)
            all_probs.extend(probs.cpu().numpy())
            all_targets.extend(labels.cpu().numpy())

    try:
        auc = roc_auc_score(all_targets, all_probs)
    except ValueError:
        auc = 0.5

    avg_loss = running_loss / len(loader)
    return avg_loss, auc

print(f"Starting training for {EPOCHS} epochs...")
best_auc = 0.0

for epoch in range(EPOCHS):
    model.train()
    train_loss = 0.0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device).float()

        optimizer.zero_grad()
        outputs = model(images).squeeze()
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()

    avg_train_loss = train_loss / len(train_loader)

    val_loss, val_auc = evaluate(model, val_loader)

    print(f"Epoch {epoch+1}/{EPOCHS} | Train Loss: {avg_train_loss:.4f} | Val Loss: {val_loss:.4f} | Val AUC: {val_auc:.4f}")


    if val_auc > best_auc:
        best_auc = val_auc
        torch.save(model.state_dict(), f"{MODEL_NAME}_best.pth")
        print(f"  >>> New Best Model Saved! (AUC: {best_auc:.4f})")

print("Training Complete!")

Writing train_image_only.py


In [ ]:
!python train_image_only.py

Using device: cpu
Preparing data for efficientnet_b5...
Downloading: "https://download.pytorch.org/models/efficientnet_b5_lukemelas-1a07897c.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b5_lukemelas-1a07897c.pth
100% 117M/117M [00:01<00:00, 83.2MB/s]
Starting training for 100 epochs...
Epoch 1/100 | Train Loss: 0.6388 | Val Loss: 0.6285 | Val AUC: 0.5750
  >>> New Best Model Saved! (AUC: 0.5750)
Traceback (most recent call last):
  File "/content/train_image_only.py", line 76, in <module>
    outputs = model(images).squeeze()
              ^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/nn/modules/module.py", line 1775, in _wrapped_call_impl
    return self._call_impl(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/nn/modules/module.py", line 1786, in _call_impl
    return forward_call(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/model.py", line 76, in forward

In [ ]:

!sed -i 's/MODEL_NAME = "efficientnet_b5"/MODEL_NAME = "inception_v3"/' train_image_only.py
!python train_image_only.py

Using device: cpu
Preparing data for inception_v3...
Downloading: "https://download.pytorch.org/models/inception_v3_google-0cc3c7bd.pth" to /root/.cache/torch/hub/checkpoints/inception_v3_google-0cc3c7bd.pth
100% 104M/104M [00:00<00:00, 190MB/s] 
Starting training for 100 epochs...
^C


In [ ]:
!sed -i 's/MODEL_NAME = "inception_v3"/MODEL_NAME = "resnet50"/' train_image_only.py
!python train_image_only.py

^C


In [ ]:
!sed -i 's/MODEL_NAME = "resnet50"/MODEL_NAME = "densenet121"/' train_image_only.py
!python train_image_only.py

^C


In [ ]:
!sed -i 's/MODEL_NAME = "densenet121"/MODEL_NAME = "mobilenet_v3_small"/' train_image_only.py
!python train_image_only.py